In [18]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression , Lasso
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# 1. CREATE SYNTHETIC DATA WITH MULTICOLLINEARITY
np.random.seed(42)
n_samples = 15000

# True area signal (what actually matters)
true_area = np.random.normal(2000, 500, n_samples)

# Create 4 highly correlated area features
Area = true_area + np.random.normal(0, 20, n_samples)
LivingArea = true_area + np.random.normal(0, 30, n_samples)
BuiltArea = true_area + np.random.normal(0, 25, n_samples)
CarpetArea = true_area + np.random.normal(0, 35, n_samples)

# Add some other features
Bedrooms = np.random.randint(1, 6, n_samples)
Bathrooms = np.random.randint(1, 4, n_samples)
Age = np.random.randint(0, 50, n_samples)

# Generate price (only truly depends on Area + others)
noise = np.random.normal(0, 50000, n_samples)
price = 200 * true_area + 50000 * Bedrooms + 30000 * Bathrooms - 1000 * Age + noise

# Create DataFrame
df = pd.DataFrame({
    'Area': Area,
    'LivingArea': LivingArea,
    'BuiltArea': BuiltArea,
    'CarpetArea': CarpetArea,
    'Bedrooms': Bedrooms,
    'Bathrooms': Bathrooms,
    'Age': Age,
    'Price': price
})

print("Correlation Matrix (Multicollinearity Check):")
print(df.head(5))
# checking the corr of the similar dataset
print(df[['Area','LivingArea','BuiltArea','CarpetArea']].corr().round(3))











Correlation Matrix (Multicollinearity Check):
          Area   LivingArea    BuiltArea   CarpetArea  Bedrooms  Bathrooms  \
0  2245.488612  2188.939917  2274.601638  2251.116398         4          1   
1  1930.214731  1899.218280  1911.354530  1950.569261         3          3   
2  2325.130167  2306.233417  2353.829360  2335.782837         5          2   
3  2780.452157  2766.004996  2759.986785  2716.787931         1          3   
4  1867.978966  1913.648182  1904.327142  1876.405602         1          3   

   Age          Price  
0   43  667781.592967  
1   44  662790.264898  
2   15  751016.114938  
3   31  675339.889431  
4    6  499487.865117  
             Area  LivingArea  BuiltArea  CarpetArea
Area        1.000       0.997      0.998       0.997
LivingArea  0.997       1.000      0.997       0.996
BuiltArea   0.998       0.997      1.000       0.996
CarpetArea  0.997       0.996      0.996       1.000


In [19]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso

X = df.drop("Price", axis=1)
y = df["Price"]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

lasso_model = Lasso(alpha=1)
lasso_model.fit(X_train, y_train)

pred_lasso = lasso_model.predict(X_test)

lasso_coefficients = pd.Series(lasso_model.coef_, index=X.columns)
print("Lasso Coefficients:")
print(lasso_coefficients)

print("Lasso R² :", r2_score(y_test, pred_lasso))
print("Lasso RMSE :",
      np.sqrt(mean_squared_error(y_test, pred_lasso)))
# now finding the best alpha value

from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Lasso

params = {
    "alpha": [0.001, 0.01, 0.1, 1, 10, 50, 100]
}

grid = GridSearchCV(
    Lasso(max_iter=10000),
    param_grid=params,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

grid.fit(X_train, y_train)

print("Best alpha:", grid.best_params_)

grid_coef = grid.best_estimator_.coef_
print("Grid Search Coefficients:")
print(grid_coef)


'''
# Lasso tends to:
# - Pick ONE feature from a correlated group
# - Assign it all the credit (large coefficient)
# - Set others to zero

# But with YOUR data:
# - 4 area features are almost perfectly correlated (r=0.997)
# - Lasso doesn't know which one is "true"
# - It splits the signal across multiple features
# - Result: No single feature gets all the credit
'''

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.277e+13, tolerance: 2.209e+10
  model = cd_fast.enet_coordinate_descent(


Lasso Coefficients:
Area          26169.975995
LivingArea    37345.411289
BuiltArea     30719.031156
CarpetArea     7418.003560
Bedrooms      70557.069057
Bathrooms     24689.679456
Age          -14028.637087
dtype: float64
Lasso R² : 0.867848794675085
Lasso RMSE : 50141.331080410906
Best alpha: {'alpha': 0.1}
Grid Search Coefficients:
[ 27174.54917709  35616.57229952  29979.78895791   8875.67586959
  70556.45104056  24689.96901418 -14029.71481372]


In [20]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

# Your data creation code (same as before)
np.random.seed(42)
n_samples = 15000
true_area = np.random.normal(2000, 500, n_samples)

Area = true_area + np.random.normal(0, 20, n_samples)
LivingArea = true_area + np.random.normal(0, 30, n_samples)
BuiltArea = true_area + np.random.normal(0, 25, n_samples)
CarpetArea = true_area + np.random.normal(0, 35, n_samples)

Bedrooms = np.random.randint(1, 6, n_samples)
Bathrooms = np.random.randint(1, 4, n_samples)
Age = np.random.randint(0, 50, n_samples)

noise = np.random.normal(0, 50000, n_samples)
price = 200 * true_area + 50000 * Bedrooms + 30000 * Bathrooms - 1000 * Age + noise

df = pd.DataFrame({
    'Area': Area,
    'LivingArea': LivingArea,
    'BuiltArea': BuiltArea,
    'CarpetArea': CarpetArea,
    'Bedrooms': Bedrooms,
    'Bathrooms': Bathrooms,
    'Age': Age,
    'Price': price
})

X = df.drop('Price', axis=1)
y = df['Price']

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ============================================
# PROPER MODEL COMPARISON
# ============================================

# 1. LINEAR REGRESSION
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)

# 2. RIDGE with GridSearch
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge())
])

ridge_params = {'ridge__alpha': [0.001, 0.01, 0.1, 1, 10, 50, 100, 500]}
ridge_grid = GridSearchCV(ridge_pipeline, ridge_params, cv=5, scoring='r2')
ridge_grid.fit(X_train, y_train)  # Pipeline handles scaling!

# 3. LASSO with GridSearch (PROPERLY SCALED)
lasso_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', Lasso(max_iter=10000))
])

lasso_params = {'lasso__alpha': [0.001, 0.01, 0.1, 1, 10, 50, 100, 500]}
lasso_grid = GridSearchCV(lasso_pipeline, lasso_params, cv=5, scoring='r2')
lasso_grid.fit(X_train, y_train)  # Pipeline handles scaling!

# ============================================
# RESULTS
# ============================================

print("=" * 60)
print("MODEL COMPARISON RESULTS")
print("=" * 60)

# Linear Regression
print(f"\nLinear Regression:")
print(f"  R²: {r2_score(y_test, lr_pred):.4f}")
print(f"  RMSE: ${np.sqrt(mean_squared_error(y_test, lr_pred)):,.0f}")

# Ridge
ridge_pred = ridge_grid.predict(X_test)
print(f"\nRidge (best alpha = {ridge_grid.best_params_['ridge__alpha']}):")
print(f"  R²: {r2_score(y_test, ridge_pred):.4f}")
print(f"  RMSE: ${np.sqrt(mean_squared_error(y_test, ridge_pred)):,.0f}")

# Lasso
lasso_pred = lasso_grid.predict(X_test)
print(f"\nLasso (best alpha = {lasso_grid.best_params_['lasso__alpha']}):")
print(f"  R²: {r2_score(y_test, lasso_pred):.4f}")
print(f"  RMSE: ${np.sqrt(mean_squared_error(y_test, lasso_pred)):,.0f}")

# ============================================
# COEFFICIENT COMPARISON
# ============================================

print("\n" + "=" * 60)
print("COEFFICIENT COMPARISON")
print("=" * 60)

# Get coefficients properly
lr_coefs = pd.Series(lr.coef_, index=X.columns)

# Ridge coefficients (from pipeline)
ridge_model = ridge_grid.best_estimator_.named_steps['ridge']
ridge_coefs = pd.Series(ridge_model.coef_, index=X.columns)

# Lasso coefficients (from pipeline)
lasso_model = lasso_grid.best_estimator_.named_steps['lasso']
lasso_coefs = pd.Series(lasso_model.coef_, index=X.columns)

coef_df = pd.DataFrame({
    'Linear': lr_coefs.round(0),
    'Ridge': ridge_coefs.round(0),
    'Lasso': lasso_coefs.round(0)
})

print(coef_df)

# ============================================
# VISUALIZE COEFFICIENTS
# ============================================

'''

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Linear
coef_df['Linear'].plot(kind='bar', ax=axes[0], color='blue', alpha=0.7)
axes[0].set_title('Linear Regression Coefficients')
axes[0].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[0].tick_params(axis='x', rotation=45)

# Ridge
coef_df['Ridge'].plot(kind='bar', ax=axes[1], color='green', alpha=0.7)
axes[1].set_title('Ridge Coefficients')
axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[1].tick_params(axis='x', rotation=45)

# Lasso
coef_df['Lasso'].plot(kind='bar', ax=axes[2], color='red', alpha=0.7)
axes[2].set_title('Lasso Coefficients')
axes[2].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()
'''
# ============================================
# WHY LASSO MIGHT NOT SHOW ZEROS
# ============================================

print("\n" + "=" * 60)
print("WHY LASSO MIGHT NOT BE SHOWING ZEROS")
print("=" * 60)

print("""
1. Alpha is too small for strong feature selection
   - Alpha=1 may not be large enough to force coefficients to zero
   - Try larger alphas: 10, 50, 100, 500

2. Data scaling is critical for Lasso
   - Lasso penalizes coefficients based on their magnitude
   - Without scaling, large-scale features dominate the penalty
   - ALWAYS scale data for Lasso!

3. Sample size is large (15,000)
   - With more data, Lasso needs larger alpha to force zeros
   - Large sample size = more evidence = harder to shrink to zero

4. Features are highly correlated (multicollinearity)
   - Lasso struggles with groups of correlated features
   - It might pick one and keep it, or keep small coefficients on all
""")

# Demonstrate with different alpha values
print("\n" + "=" * 60)
print("LASSO COEFFICIENTS AT DIFFERENT ALPHAS")
print("=" * 60)

alphas_to_test = [0.01, 0.1, 1, 10, 50, 100, 500]
lasso_results = {}

for alpha in alphas_to_test:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train_scaled, y_train)
    lasso_results[alpha] = lasso.coef_

# Display results
for alpha in alphas_to_test:
    coefs = pd.Series(lasso_results[alpha], index=X.columns)
    zeros = (coefs.abs() < 1).sum()  # Count near-zero coefficients
    print(f"\nAlpha = {alpha}:")
    print(coefs.round(0).to_string())
    print(f"  Features with coefficients near zero: {zeros}")





MODEL COMPARISON RESULTS

Linear Regression:
  R²: 0.8679
  RMSE: $50,138

Ridge (best alpha = 10):
  R²: 0.8679
  RMSE: $50,137

Lasso (best alpha = 1):
  R²: 0.8679
  RMSE: $50,138

COEFFICIENT COMPARISON
             Linear    Ridge    Lasso
Area        27157.0  26932.0  27184.0
LivingArea  35613.0  33915.0  35610.0
BuiltArea   29994.0  29221.0  29973.0
CarpetArea   8882.0  11557.0   8879.0
Bedrooms    70557.0  70495.0  70556.0
Bathrooms   24690.0  24668.0  24689.0
Age        -14030.0 -14017.0 -14029.0

WHY LASSO MIGHT NOT BE SHOWING ZEROS

1. Alpha is too small for strong feature selection
   - Alpha=1 may not be large enough to force coefficients to zero
   - Try larger alphas: 10, 50, 100, 500

2. Data scaling is critical for Lasso
   - Lasso penalizes coefficients based on their magnitude
   - Without scaling, large-scale features dominate the penalty
   - ALWAYS scale data for Lasso!

3. Sample size is large (15,000)
   - With more data, Lasso needs larger alpha to force zeros
